In [2]:
# iterate through all files (guaranteed to be .csv) in the directory "../../data/clean/subtitles"
# and print out all of their pandas dataframe sizes
import os
import pandas as pd



subtitle_dir = "../../data/clean/subtitles/"

total_time = 0

for file in os.listdir(subtitle_dir):
    if file.endswith(".csv"):
        df = pd.read_csv(subtitle_dir + file)
        # print(file, df.shape)
        length = df["end"].iloc[-1]
        total_time += length
        print(file, length)
print(total_time)

0.csv 246.288
1.csv 198.912
2.csv 102.0
3.csv 242.088
4.csv 220.584
5.csv 232.248
6.csv 210.504
7.csv 217.08
8.csv 243.096
9.csv 255.12
10.csv 216.264
11.csv 245.064
12.csv 283.176
13.csv 252.288
14.csv 205.752
15.csv 178.752
16.csv 220.8
17.csv 289.032
18.csv 154.2
19.csv 220.824
20.csv 108.504
21.csv 178.08
22.csv 246.936
23.csv 225.048
24.csv 181.848
25.csv 131.232
26.csv 205.824
28.csv 170.352
29.csv 155.952
30.csv 129.048
31.csv 304.992
32.csv 270.048
33.csv 265.056
34.csv 241.968
35.csv 280.08
36.csv 263.256
37.csv 102.048
38.csv 265.032
39.csv 262.776
40.csv 286.632
41.csv 255.696
43.csv 316.152
44.csv 273.312
45.csv 324.648
47.csv 276.672
48.csv 276.024
49.csv 240.456
50.csv 272.184
51.csv 218.184
53.csv 129.984
55.csv 317.4
56.csv 253.968
58.csv 265.752
64.csv 250.056
67.csv 204.792
70.csv 305.04
74.csv 299.088
78.csv 107.28
79.csv 101.88
81.csv 109.824
83.csv 96.912
85.csv 260.352
89.csv 256.56
90.csv 235.152
91.csv 236.088
92.csv 202.584
14794.823999999999


In [27]:
# do the same, but for "../../data/clean/segment_breaks"
length_data = []
number_less = 0
total_column_count = 0
number_less_df = None
number_less_dfs = []
subtitle_dir = "../../data/clean/subtitles/"
for file in os.listdir(subtitle_dir):
    if file.endswith(".csv"):
        df = pd.read_csv(subtitle_dir + file)
        # 'token' is not '<silence>'
        df = df[df["token"] != "<silence>"]
        # numerical summary of end - start
        length_description = df["end"].sub(df["start"]).describe()
        length_data.append((file, length_description))
        # print(file, length_description)
        # percent of all rows with end - start < 0.010
        # append to number_less_dfs the rows of the dataframe that have end - start < 0.010
        #if (df["end"].sub(df["start"]) < 0.010).sum() > 0:
        #    number_less_dfs.append(df[df["end"].sub(df["start"]) < 0.010])
        number_less += (df["end"].sub(df["start"]) < 0.010).sum()
        total_column_count = df.shape[0]
print(f"{number_less/total_column_count*100}%")
#number_less_df = pd.concat(number_less_dfs)

0.0%


In [28]:
# take the lowest, median, and highest of each entry of the length description across all files
def get_stat(data):
    means, stds, mins, first_qs, medians, third_qs, maxs = [], [], [], [], [], [], []
    for _, description in data:
        means.append(description["mean"])
        stds.append(description["std"])
        mins.append(description["min"])
        first_qs.append(description["25%"])
        medians.append(description["50%"])
        third_qs.append(description["75%"])
        maxs.append(description["max"])
    # lowest of all
    min_min = min(mins)
    print(f"min: {min_min}")
    max_max = max(maxs)
    print(f"max: {max_max}")
    
    # find median of medians
    medians = sorted(medians)
    median_median = medians[len(medians) // 2]
    print(f"list of medians: {median_median}")
    # find median of means
    means = sorted(means)
    median_mean = means[len(means) // 2]
    print(f"list of means: {median_mean}")
    # min of first quartiles
    min_first_q = min(first_qs)
    print(f"min of first quartiles: {min_first_q}")
get_stat(length_data)

min: 0.03299999999998704
max: 6.1059999999999945
list of medians: 0.2330000000000041
list of means: 0.3029703703703704
min of first quartiles: 0.10000000000000142


In [24]:
number_less_df

,start,end,token,exclude
837,233.21,233.217,<silence>,False


# Song Similarities

In [1]:
import librosa
import numpy as np
from sklearn.metrics.pairwise import cosine_distances
from pathlib import Path

def get_unique_songs(directory, n_songs=5):
    features = []
    files = []

    # Extract features
    for mp3 in Path(directory).glob('*.mp3'):
        y, sr = librosa.load(str(mp3))
        mfcc = librosa.feature.mfcc(y=y, sr=sr)
        features.append(np.mean(mfcc, axis=1))
        files.append(mp3)

    # Calculate distances
    distances = cosine_distances(features)

    # Get the most unique (highest average distance to others)
    avg_distances = np.mean(distances, axis=1)
    unique_indices = np.argsort(avg_distances)[-n_songs:]

    return [files[i] for i in unique_indices]

In [2]:
unique_songs = get_unique_songs('../../data/clean/audio/vocals')

In [3]:
unique_songs

[PosixPath('../../data/clean/audio/vocals/19.mp3'),
 PosixPath('../../data/clean/audio/vocals/85.mp3'),
 PosixPath('../../data/clean/audio/vocals/60.mp3'),
 PosixPath('../../data/clean/audio/vocals/86.mp3'),
 PosixPath('../../data/clean/audio/vocals/21.mp3')]

# Word Frequencies

In [7]:
import pandas as pd
from pathlib import Path

def analyze_tokens(token_set, csv_directory):
    # Convert token set to lowercase for case-insensitive comparison
    token_set = {t.lower() for t in token_set}

    # Initialize results structure
    # {token: {filename: count}}
    token_counts = {token: {} for token in token_set}
    missing_tokens = set()

    # Iterate through CSV files in directory
    for file_path in Path(csv_directory).glob('*.csv'):
        try:
            df = pd.read_csv(file_path)

            if 'token' not in df.columns:
                print(f"Warning: No 'token' column in {file_path}")
                continue

            # Convert tokens to lowercase
            file_tokens = df['token'].astype(str).str.lower()

            # Count occurrences of each token in this file
            for token in token_set:
                count = sum(file_tokens == token)
                if count > 0:
                    token_counts[token][file_path.name] = count

            # Find tokens not in token_set
            missing_tokens.update(
                set(file_tokens.unique()) - token_set
            )

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    return token_counts, missing_tokens

In [8]:
from data_processing.utils.tokens import get_tokens
token_set = get_tokens('../../data/config/tokens.txt')
csv_directory = '../../data/clean/subtitles/'

token_counts, missing_tokens = analyze_tokens(token_set, csv_directory)


Counts per token per file:

:
  Not found in any file

ど:
  0.csv: 4
  1.csv: 11
  2.csv: 5
  3.csv: 3
  4.csv: 7
  5.csv: 6
  6.csv: 8
  7.csv: 5
  8.csv: 5
  9.csv: 8
  10.csv: 5
  11.csv: 3
  12.csv: 1
  13.csv: 8
  15.csv: 2
  16.csv: 3
  17.csv: 3
  18.csv: 8
  19.csv: 5
  20.csv: 2
  22.csv: 1
  24.csv: 1
  25.csv: 9
  26.csv: 5
  28.csv: 10
  29.csv: 6
  30.csv: 4
  31.csv: 3
  32.csv: 1
  33.csv: 8
  34.csv: 4
  35.csv: 5
  36.csv: 8
  37.csv: 1
  38.csv: 7
  39.csv: 8
  40.csv: 3
  41.csv: 3
  43.csv: 4
  44.csv: 6
  45.csv: 4
  47.csv: 3
  48.csv: 2
  49.csv: 3
  50.csv: 16
  51.csv: 6
  53.csv: 2
  55.csv: 1
  56.csv: 5
  58.csv: 8
  64.csv: 8
  67.csv: 6
  70.csv: 3
  74.csv: 4
  78.csv: 3
  79.csv: 3
  81.csv: 4
  83.csv: 1
  85.csv: 2
  89.csv: 8
  90.csv: 6
  91.csv: 2
  92.csv: 5

じょ:
  3.csv: 1
  8.csv: 1
  10.csv: 1
  13.csv: 5
  15.csv: 1
  17.csv: 2
  19.csv: 2
  23.csv: 1
  25.csv: 6
  26.csv: 2
  33.csv: 1
  34.csv: 3
  35.csv: 1
  37.csv: 2
  41.csv: 1
  43.csv:

In [19]:
# find which file contains chinese tokens (it's part of the missing tokens)
# search by unicode range

# unicode range for chinese characters
chinese_range = (0x4E00, 0x9FFF)

# iterate through all missing chinese tokens, and tell us which files they are in;
# example, {<chinese token>: [<file1>, <file2>, ...]}
chinese_tokens = {}
# search through all the files again
for file_path in Path(csv_directory).glob('*.csv'):
    try:
        df = pd.read_csv(file_path)
        if 'token' not in df.columns:
            print(f"Warning: No 'token' column in {file_path}")
            continue
        # find all chinese tokens in the file
        file_tokens = df['token'].astype(str)
        for token in missing_tokens:
            if any(ord(char) >= chinese_range[0] and ord(char) <= chinese_range[1] for char in token):
                if token not in chinese_tokens:
                    chinese_tokens[token] = []
                chinese_tokens[token].append(file_path.name)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        
chinese_tokens

{'綺': ['0.csv',
  '1.csv',
  '2.csv',
  '3.csv',
  '4.csv',
  '5.csv',
  '6.csv',
  '7.csv',
  '8.csv',
  '9.csv',
  '10.csv',
  '11.csv',
  '12.csv',
  '13.csv',
  '14.csv',
  '15.csv',
  '16.csv',
  '17.csv',
  '18.csv',
  '19.csv',
  '20.csv',
  '21.csv',
  '22.csv',
  '23.csv',
  '24.csv',
  '25.csv',
  '26.csv',
  '28.csv',
  '29.csv',
  '30.csv',
  '31.csv',
  '32.csv',
  '33.csv',
  '34.csv',
  '35.csv',
  '36.csv',
  '37.csv',
  '38.csv',
  '39.csv',
  '40.csv',
  '41.csv',
  '43.csv',
  '44.csv',
  '45.csv',
  '47.csv',
  '48.csv',
  '49.csv',
  '50.csv',
  '51.csv',
  '53.csv',
  '55.csv',
  '56.csv',
  '58.csv',
  '64.csv',
  '67.csv',
  '70.csv',
  '74.csv',
  '78.csv',
  '79.csv',
  '81.csv',
  '83.csv',
  '85.csv',
  '89.csv',
  '90.csv',
  '91.csv',
  '92.csv'],
 '持': ['0.csv',
  '1.csv',
  '2.csv',
  '3.csv',
  '4.csv',
  '5.csv',
  '6.csv',
  '7.csv',
  '8.csv',
  '9.csv',
  '10.csv',
  '11.csv',
  '12.csv',
  '13.csv',
  '14.csv',
  '15.csv',
  '16.csv',
  '17.csv',
 

In [9]:
print("\nToken counts:")
for token in sorted(token_set):
    counts_str = ", ".join(f"({file}, {count})" for file, count in token_counts[token].items())
    print(f"{token}: {counts_str if counts_str else 'not found'}")

print("\nTokens not in set:", ", ".join(sorted(missing_tokens)))


Token counts:
: not found
<silence>: (0.csv, 11), (1.csv, 15), (2.csv, 4), (3.csv, 8), (4.csv, 12), (5.csv, 9), (6.csv, 13), (7.csv, 10), (8.csv, 13), (9.csv, 11), (10.csv, 9), (11.csv, 8), (12.csv, 38), (13.csv, 15), (14.csv, 10), (15.csv, 8), (16.csv, 15), (17.csv, 7), (18.csv, 5), (19.csv, 8), (20.csv, 5), (21.csv, 6), (22.csv, 13), (23.csv, 13), (24.csv, 6), (25.csv, 8), (26.csv, 10), (28.csv, 14), (29.csv, 7), (30.csv, 10), (31.csv, 17), (32.csv, 6), (33.csv, 21), (34.csv, 14), (35.csv, 6), (36.csv, 13), (37.csv, 5), (38.csv, 7), (39.csv, 6), (40.csv, 20), (41.csv, 8), (43.csv, 15), (44.csv, 12), (45.csv, 7), (47.csv, 7), (48.csv, 5), (49.csv, 11), (50.csv, 19), (51.csv, 7), (53.csv, 15), (55.csv, 5), (56.csv, 11), (58.csv, 6), (64.csv, 7), (67.csv, 11), (70.csv, 16), (74.csv, 7), (78.csv, 8), (79.csv, 21), (81.csv, 3), (83.csv, 6), (85.csv, 15), (89.csv, 7), (90.csv, 18), (91.csv, 9), (92.csv, 11)
あ: (0.csv, 13), (1.csv, 5), (2.csv, 5), (3.csv, 5), (4.csv, 13), (5.csv, 12), (6.c

In [28]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import natsort
import numpy as np


def analyze_tokens_to_heatmap(token_set, csv_directory, blue_scale_files=None, output_path='heatmap.png'):
    # Set font that supports Japanese characters
    plt.rcParams['font.family'] = 'Noto Sans CJK JP'

    token_set = {t.lower() for t in token_set}

    # Use natsort to properly sort filenames
    files = natsort.natsorted([f.name.split('.')[0] for f in Path(csv_directory).glob('*.csv')])
    counts_df = pd.DataFrame(0,
                             index=sorted(token_set),
                             columns=files)

    for file_path in Path(csv_directory).glob('*.csv'):
        try:
            df = pd.read_csv(file_path)
            if 'token' not in df.columns:
                print(f"Warning: No 'token' column in {file_path}")
                continue

            file_tokens = df['token'].astype(str).str.lower()
            file_name = file_path.name.split('.')[0]

            for token in token_set:
                count = sum(file_tokens == token)
                counts_df.loc[token, file_name] = count

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    # Create large figure with adjusted height
    fig, ax = plt.subplots(figsize=(20, max(5, len(token_set) * 0.3)))

    # Convert DataFrame to numpy for plotting
    data = counts_df.values

    # Create a mask for blue scale columns
    blue_scale_mask = np.zeros_like(data, dtype=bool)
    if blue_scale_files:
        for j, col in enumerate(files):
            if col in blue_scale_files:
                blue_scale_mask[:, j] = True

    # Plot the heatmap
    im = ax.imshow(data,
                   cmap='YlOrRd',  # Default colormap
                   aspect='auto',
                   interpolation='nearest')

    # Add text annotations
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            # Customize color based on background intensity
            val = data[i, j]
            color = 'white' if val > data.max() * 0.6 else 'black'
            ax.text(j, i, int(val),
                    ha='center', va='center',
                    color=color)

    # Customize the display
    ax.set_xticks(np.arange(len(files)))
    ax.set_yticks(np.arange(len(token_set)))
    ax.set_xticklabels(files, rotation=45, ha='right')
    ax.set_yticklabels(list(counts_df.index))

    # Overlay blue colormap for specified columns
    if blue_scale_files:
        # Create a copy of the data to modify
        modified_data = data.copy().astype(float)

        # Apply blue colormap to specified columns
        blue_cmap = plt.cm.Blues
        for j, col in enumerate(files):
            if col in blue_scale_files:
                column_data = data[:, j]
                norm = mcolors.Normalize(vmin=column_data.min(), vmax=column_data.max())
                modified_data[:, j] = norm(column_data)

        # Create a new imshow with the modified data and blue colormap
        im_blue = ax.imshow(modified_data,
                            cmap=blue_cmap,
                            alpha=0.7,  # Adjust transparency
                            aspect='auto',
                            interpolation='nearest')

    plt.title('Token Counts per File', fontsize=16)
    plt.ylabel('Tokens', fontsize=12)
    plt.xlabel('Files', fontsize=12)

    # Add colorbar
    plt.colorbar(im)

    # Adjust layout to prevent cutoff
    plt.tight_layout()

    # Save high-resolution figure
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

    return output_path

# Specify which files you want in blue scale
blue_scale_files = ['0', '6', '16']

output_file = analyze_tokens_to_heatmap(token_set, csv_directory, blue_scale_files)
print(f"Heatmap saved as: {output_file}")


Heatmap saved as: heatmap.png


In [22]:
import pandas as pd
import os
from pathlib import Path
import re

def find_chinese_tokens_locations(token_set, csv_directory):
    token_set = {t.lower() for t in token_set}
    chinese_token_files = {}

    # Function to check if a string contains Chinese characters
    def has_chinese(text):
        # Unicode ranges for Chinese characters
        chinese_ranges = [
            (0x4E00, 0x9FFF),   # CJK Unified Ideographs
            (0x3400, 0x4DBF),   # CJK Unified Ideographs Extension A
            (0x20000, 0x2A6DF), # CJK Unified Ideographs Extension B
            (0x2A700, 0x2B73F), # CJK Unified Ideographs Extension C
            (0x2B740, 0x2B81F), # CJK Unified Ideographs Extension D
            (0x2B820, 0x2CEAF), # CJK Unified Ideographs Extension E
            (0x2CEB0, 0x2EBEF), # CJK Unified Ideographs Extension F
        ]
        for char in text:
            code = ord(char)
            if any(start <= code <= end for start, end in chinese_ranges):
                return True
        return False

    for file_path in Path(csv_directory).glob('*.csv'):
        try:
            df = pd.read_csv(file_path)

            if 'token' not in df.columns:
                print(f"Warning: No 'token' column in {file_path}")
                continue

            file_tokens = df['token'].astype(str).str.lower()

            # Find Chinese tokens in this file that aren't in token_set
            chinese_tokens = {token for token in file_tokens.unique()
                              if has_chinese(token) and token not in token_set}

            # Add file to the list for each Chinese token
            for token in chinese_tokens:
                if token not in chinese_token_files:
                    chinese_token_files[token] = []
                chinese_token_files[token].append(file_path.name)

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    return chinese_token_files

# Example usage:
chinese_tokens = find_chinese_tokens_locations(token_set, csv_directory)

# Print results
print("\nChinese tokens and their files:")
for token, files in sorted(chinese_tokens.items()):
    print(f"{token}: {files}")


Chinese tokens and their files:
超: ['2.csv']
